In [ ]:
!pip install transformers datasets evaluate seqeval

In [ ]:
!pip install -U transformers

In [ ]:
!pip install -U datasets

In [1]:
import json

def load_model_group_dataset(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    processed = []

    for item in data:
        text = item.get("text")
        label = item.get("model_group")

        if text and label:
            processed.append({
                "text": text,
                "label": label
            })

    return processed


dataset_raw = load_model_group_dataset("meged3.json")

print(f"Loaded {len(dataset_raw)} examples for model_group")
print(dataset_raw[0])

Loaded 3223 examples for model_group
{'text': 'Ғалым бұл ұғым өнер тарихы, археология, мәдениеттану, антропология, этнология және дінтану салаларымен байланысты болғандықтан, гуманитарлық пәндер қатарында қосуды ұсынды.', 'label': 'SYNTHETIC'}


In [2]:
from sklearn.preprocessing import LabelEncoder

texts = [x["text"] for x in dataset_raw]
labels = [x["label"] for x in dataset_raw]

label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

num_labels = len(label_encoder.classes_)
print("Number of classes:", num_labels)
print("Classes:", list(label_encoder.classes_))

id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in enumerate(label_encoder.classes_)}

Number of classes: 3
Classes: [np.str_('ANALYTIC'), np.str_('ANALYTICO_SYNTHETIC'), np.str_('SYNTHETIC')]


In [3]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    encoded_labels,
    test_size=0.2,
    random_state=42,
    stratify=encoded_labels
)

train_dataset = Dataset.from_dict({
    "text": train_texts,
    "labels": train_labels
})

test_dataset = Dataset.from_dict({
    "text": test_texts,
    "labels": test_labels
})

c:\Users\egisb\voice1\kazFineTune\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [5]:
from transformers import AutoTokenizer

model_ckpt = "Eraly-ml/KazBERT"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True, remove_columns=["text"])
test_dataset = test_dataset.map(tokenize, batched=True, remove_columns=["text"])

Map: 100%|██████████| 645/645 [00:00<00:00, 4254.29 examples/s]


In [6]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at Eraly-ml/KazBERT and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    labels = p.label_ids

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

In [2]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./kazbert_model_group_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    logging_dir="./logs"
)

c:\Users\egisb\voice1\kazFineTune\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

C:\Users\egisb\AppData\Local\Temp\ipykernel_34596\4223154140.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.023880,0.995349,0.995467
2,No log,0.021265,0.995349,0.995471
3,No log,0.013822,0.996899,0.997446
4,0.071300,0.012831,0.996899,0.997446
5,0.071300,0.015577,0.996899,0.997446
6,0.071300,0.016254,0.996899,0.997446
7,0.000400,0.015964,0.996899,0.997446
8,0.000400,0.016004,0.996899,0.997446


TrainOutput(global_step=1296, training_loss=0.027705892759524745, metrics={'train_runtime': 2654.1157, 'train_samples_per_second': 7.771, 'train_steps_per_second': 0.488, 'total_flos': 1356612781805568.0, 'train_loss': 0.027705892759524745, 'epoch': 8.0})

In [1]:
results = trainer.evaluate()
print(results)

NameError: name 'trainer' is not defined

In [11]:
from sklearn.metrics import classification_report

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

print(classification_report(
    test_labels,
    preds,
    target_names=label_encoder.classes_
))

                     precision    recall  f1-score   support

           ANALYTIC       0.99      1.00      1.00       245
ANALYTICO_SYNTHETIC       1.00      1.00      1.00       121
          SYNTHETIC       1.00      0.99      1.00       279

           accuracy                           1.00       645
          macro avg       1.00      1.00      1.00       645
       weighted avg       1.00      1.00      1.00       645



In [12]:
trainer.save_model("kazbert_group_model_final")
tokenizer.save_pretrained("kazbert_group_model_final")

('kazbert_group_model_final\\tokenizer_config.json',
 'kazbert_group_model_final\\special_tokens_map.json',
 'kazbert_group_model_final\\vocab.txt',
 'kazbert_group_model_final\\added_tokens.json',
 'kazbert_group_model_final\\tokenizer.json')

In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

model_dir = "kazbert_group_model_final"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

model_group_pipeline = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True
)

Device set to use cuda:0
c:\Users\egisb\voice1\kazFineTune\.venv\lib\site-packages\transformers\pipelines\text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [14]:
sentence = "Білім беру стандарты жаңартылмай, оқу мазмұны өзгертілмеді."
prediction = model_group_pipeline(sentence)

print(prediction)

[[{'label': 'ANALYTIC', 'score': 9.519554441794753e-05}, {'label': 'ANALYTICO_SYNTHETIC', 'score': 0.00014095916412770748}, {'label': 'SYNTHETIC', 'score': 0.9997639060020447}]]
